# Advanced Python Programming
This notebook covers the fundamentals of **generators**,  **lazy evaluation**, and many more. This week, we focus on designing and applying **decorators** for function wrapping and behavioural extension. We build and use **context managers** for safe resource handling and apply **functional programming concepts** in Python. Each section includes explanations and examples.

# 1. Generators and Lazy Evaluation
### Conceptual Overview
A **generator** is a function that:
- Uses `yield` instead of `return`
- Produces values **lazily (on demand)**
- Maintains internal state between iterations
- Is memory-efficient for large datasets

**Lazy evaluation** means:
- Values are computed **only** when **requested**
- No full materialisation in memory
- Particularly useful for:
  - Large files
  - Streams
  - Infinite sequences

In [1]:
# Defines a generator function
def even_numbers(limit): 
    for i in range(limit):
        if i % 2 == 0:
            ''' Instead of returning a value and exiting, 
            this yields the current even number and pauses execution until the next value is requested.'''
            yield i 
# Using the generator function
for num in even_numbers(10):
    print(num)

0
2
4
6
8


Let's have a look at this simple example:
 ```python
 our_range=range(1000000000)
 ```

**Python does not create one billion integers in memory. Instead it stores only:**
```python
start=0
stop=1000000000
step=1
```

when you ask for an element:
```python 
print(our_range[5]) --> 5 
```
**Python computes that value on demand.**

## List Comprehension vs Generator Expression

### Key Differences

| Feature | List Comprehension | Generator Expression |
|---|---|---|
| **Returns** | `list` object| `generator` object |
| **Evaluation** | Eager (all at once) | Lazy (one at a time) |
| **Memory** | Stores all values in RAM | Stores only current state |
| **Reusable** | Yes, iterate multiple times | No, **exhausted after one pass** |
| **Speed (first item)** | Slower (builds entire list first) | Faster (yields immediately) |


In [12]:
# Generator Expression - Like list comprehensions, but lazy.

# List comprehension is eager, **loads all into memory**  Syntax: square brackets
squares_list = [x**2 for x in range(1_000_000)] # list object

# Generator expression is lazy, **computes on demand** Syntax: parentheses
squares_gen = (x**2 for x in range(1_000_000)) # generator object

# Memory comparison
import sys
print(sys.getsizeof(squares_list))  # ~8 MB
print(sys.getsizeof(squares_gen))   # ~200 bytes


8448728
200


In [3]:
# Reusability in Generator expression
gen_object = (x**2 for x in range(5))
print(list(gen_object))  # [0, 1, 4, 9, 16]
print(list(gen_object))  # [] ← exhausted! Can't reuse.

[0, 1, 4, 9, 16]
[]


In [17]:
# Reusability in List comprehension
list_object = [x**2 for x in range(5)]
print(list_object)  
print(list_object) 

[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]


In [47]:
# Reading a Large File with a Generator 
'''The generator approach is memory-efficient 
    because it only loads one line at a time, which is ideal for very large files.
'''
def read_large_file(filepath, encoding="utf-8"):
    """Yield lines from a file without loading it all into memory."""
    with open(filepath, "r", encoding=encoding) as f:  # Added encoding parameter here
        for line in f:
            yield line.strip()

lines = list(read_large_file("data.csv", encoding="utf-8"))
print(lines)

['name,age,city', 'Alice,25,Berlin', 'Bob,30,Hamburg', 'Charlie,28,Munich', 'Diana,22,Frankfurt', 'Edward,35,Cologne', 'Fatima,27,Stuttgart', 'George,40,Düsseldorf', 'Hannah,29,Leipzig', 'Ivan,31,Dresden', 'Julia,26,Bremen', 'Karim,33,Hanover', 'Laura,24,Nuremberg', 'Michael,38,Berlin', 'Nina,21,Hamburg', 'Omar,36,Munich', 'Paula,23,Frankfurt', 'Quentin,41,Cologne', 'Rania,28,Stuttgart', 'Sam,34,Düsseldorf', 'Tina,27,Leipzig', 'Umar,32,Dresden', 'Valeria,25,Bremen', 'William,37,Hanover', 'Xenia,29,Nuremberg', 'Yusuf,30,Berlin', 'Zara,22,Hamburg', 'Adam,39,Munich', 'Bianca,31,Frankfurt', 'Carlos,33,Cologne', 'Daniela,26,Stuttgart', 'Elias,28,Düsseldorf', 'Farah,35,Leipzig', 'Gustav,42,Dresden', 'Helena,24,Bremen', 'Ismail,29,Hanover', 'Jonas,34,Nuremberg', 'Klara,23,Berlin', 'Lars,38,Hamburg', 'Maya,27,Munich', 'Noah,31,Frankfurt', 'Olivia,26,Cologne', 'Petra,40,Stuttgart', 'Rashid,36,Düsseldorf', 'Sara,28,Leipzig', 'Tom,33,Dresden']


In [23]:
# yield from for Sub-Generators
###################################
'''
When the function encounters a list item during iteration,
instead of yielding the list itself,
it delegates to another generator (a recursive call to *flatten(item)*).
'''
###################################
def flatten(nested):
    """Recursively flatten a nested iterable."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item) # useful for recursive generator functions like this flattening 
        else:
            yield item

data = [1, [2, [3, 4]], [5, 6]]
print(list(flatten(data)))  

[1, 2, 3, 4, 5, 6]


An **infinite generator** is a generator that can produce values indefinitely.  
Unlike normal functions that eventually terminate, an infinite generator continues producing values whenever the next value is requested.

This is useful when:

- Generating continuous data streams
- Producing IDs
- Simulating real-time data
- Working with lazy evaluation pipelines

Because the generator never stops, it is important to **control iteration externally** (for example with `break`, `islice`, or a condition).


In [24]:
# Infinite Generators
###################################
'''
This code demonstrates how to create an infinite generator for even numbers 
and how to extract a finite sequence from it
'''
###################################
def even_numbers():
    n = 0
    while True:
        yield n
        n += 2
gen = even_numbers()

for _ in range(5):
    print(next(gen))

0
2
4
6
8


**Visual execution flow (Illusrative example)**
> Note: the numbers will be different; however, this visual execution flow is provided as a proof of concept.
```python
gen = random_numbers()
```
```
+---------------------------+
| STATE 0: generator created |
+---------------------------+
| no computation yet        |
+---------------------------+
            ↓
+---------------------------+
| STATE 1: next(gen)       |
+---------------------------+
| compute: 83              |
| yield: 83                |
| state saved              |
+---------------------------+
            ↓
+---------------------------+
| STATE 2: next(gen)       |
+---------------------------+
| compute: 48             |
| yield: 48               |
| state saved             |
+---------------------------+
            ↓
+---------------------------+
| STATE 3: next(gen)       |
+---------------------------+
| compute: 91              |
| yield: 91                |
| state saved              |
+---------------------------+
           ↓
+---------------------------+
| STATE 4: next(gen)       |
+---------------------------+
| compute: 64              |
| yield: 64                |
| state saved              |
+---------------------------+
           ↓
+---------------------------+
| STATE 5: next(gen)       |
+---------------------------+
| compute: 93              |
| yield: 93                |
| state saved              |
+---------------------------+
          ↓
+---------------------------+
|        DONE               |
+---------------------------+
```

In [ ]:
import random  # Provides functions for generating random values.

# Infinite generator function
def random_numbers():
    """
    Generates random integers between 1 and 100 indefinitely.

    The function uses `yield`, which makes it a generator.
    Each call to `next()` produces one random number and then
    pauses execution until the next request.
    """
    
    while True:  # Infinite loop: the generator can produce unlimited values.
        
        # Generate a random integer between 1 and 100,
        # return it to the caller, and pause execution.
        yield random.randint(1, 100)


# Create a generator object.
# No random numbers are generated at this point.
gen = random_numbers()

# Request and print five random numbers from the generator.
# _ conventional throwaway variable, you replace the variable name with _ to indicate:
# **I do not care about the loop index**
for _ in range(5):
    
    # `next(gen)` resumes the generator,
    # generates the next random number,
    # and returns it.
    print(next(gen))

83
48
91
64
93


In [4]:
###################################
'''
Because infinite generators never stop, 
we often limit them using itertools.islice
'''
###################################

from itertools import islice

def infinite_counter():
    n = 0
    while True:
        yield n
        n += 1
'''
 Uses `islice` to limit the infinite generator to only its first 10 values. 
 This prevents an infinite loop while still allowing us to work with the infinite generator.
'''
for value in islice(infinite_counter(), 10):
    print(value)

0
1
2
3
4
5
6
7
8
9


# 2. Decorators and Function Wrapping
### Key Concept
A **decorator** is a higher-order construct that enables the modification or enhancement of a function’s behaviour **without altering its source code directly**. This is achieved by **function wrapping**, where one function takes another function as input, extends or modifies its behaviour, and returns a new callable.

They allow:

- Cross-cutting concerns (logging, timing, authentication)

- Behaviour injection

- Code reuse without inheritance

- Transparent extension

A basic decorator example:
```python 
 def my_decorator(func):
    def wrapper():
        print("Before function execution")
        func()
        print("After function execution")
    return wrapper
@my_decorator
def greet():
    print("Hello")
```

In [14]:
def my_decorator(func):
    def wrapper():
        print("Before function execution")
        func()
        print("After function execution")
    return wrapper
@my_decorator
def greet():
    print("Hello")
greet()

Before function execution
Hello
After function execution


### Function Wrapping Mechanism

The wrapper function is the key mechanism doing the following:
- intercepts the call to the original function
- optionally modifies inputs (args, kwargs)
- executes additional logic before and after
- returns or alters the original output

This pattern is often described as **aspect-oriented programming behaviour**, where cross-cutting concerns are separated from core logic.

```python 
 def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator
@repeat(3)
def hello():
    print("Hi")
```

In [21]:
#############################
# Parameterised Decorator 
#############################
# Outer function:
# receives the decorator argument (n = number of repetitions)
def repeat(n):
    # Actual decorator:
    # receives the function being decorated
    def decorator(func):
        # Wrapper function:
        # *args  collects all positional arguments into a tuple
        # **kwargs collects all keyword arguments into a dictionary
        #
        # Using *args and **kwargs makes the decorator work with
        # any function signature, not just functions with no arguments.
        def wrapper(*args, **kwargs):
            # Repeat the function call n times
            for _ in range(n): # underscore `_` is used as a variable name by convention when the loop variable itself is not needed within the loop body34
                # Forward all received arguments to the original function
                func(*args, **kwargs) 
        # Return the wrapped version of the function
        return wrapper
    # Return the decorator
    return decorator
# Equivalent to:
# hello = repeat(3)(hello)
@repeat(3)
def hello():
    print("Hi")
hello()

Hi
Hi
Hi


> Have a look at this [link](https://docs.python.org/3/library/functools.html#functools.wraps) before running next code.

In [22]:
###############################################################
# Basic Decorator with Metadata Preservation                  
# This code defines a decorator called `log_call`              
# that adds logging functionality to any function it decorates. 
###############################################################
from functools import wraps # helps preserve the metadata of the decorated function

def log_call(func):
    @wraps(func) # This decorator preserves the original function's metadata (like name, docstring, etc.) in the wrapped function
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Returned {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

# When called:
result = add(3, 5)
# Output:
# Calling add
# Returned 8

Calling add
Returned 8


In [45]:
#############################
# Class-Based Decorator 
#This decorator allows you to track how many times a function has been called 
# while preserving the function's original behaviour.
#############################
class CountCalls:
    """
    - The constructor method that initialises:
    - `self.func = func` - Stores the function being decorated
    - `self.count = 0` - Initialises a counter to track how many times the function is called
    """
    def __init__(self, func):
        self.func = func
        self.count = 0
    """
    This special method makes instances of the class callable:
       -`*args` collects all positional arguments passed to the decorated function
       - `**kwargs` collects all keyword arguments passed to the decorated function
    """
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call number {self.count}")
        return self.func(*args, **kwargs)
        
@CountCalls
def greet(name):
    print(f"Hello {name}")
'''Python internally performs --> greet = CountCalls(greet)'''
greet('Alex')
greet('Adam')
greet('Sue')

@CountCalls
def square(x):
    return x * x
print(square(2))
print(square(3))
print(square(4))
print(square(5))
'''
Since the decorated function is now an object, we can access its attributes.
'''
print('Count is : ', square.count) # Accessing the Call Counter

Call number 1
Hello Alex
Call number 2
Hello Adam
Call number 3
Hello Sue
Call number 1
4
Call number 2
9
Call number 3
16
Call number 4
25
Count is :  4


# 3. Context Managers and Resource Handling
A context manager ensures that setup and teardown code always runs, even in the presence of exceptions. The `with` statement is the standard interface, and it underpins safe file handling, database transactions, locks, and more.
### Context Managers Matter

**Resources** include:

- Files

- Database connections

- Network sockets

- Locks

- Temporary directories

**Failure** to release resources leads to:

- Memory leaks

- File descriptor exhaustion

- Deadlocks

- Data corruption

**Understanding the Protocol**

A context manager usually implements, or its equivalent:

- ```__enter__```

- ``` __exit__```

In [51]:
class ManagedResource:
    def __enter__(self):
        print("Acquiring resource")
        return self          # Value bound to `as` variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Releasing resource")
        # Return True to suppress exceptions, False (or None) to propagate
        return False

with ManagedResource() as r:
    print("Using resource")

Acquiring resource
Using resource
Releasing resource


In [53]:
import time

class Timer:
    def __enter__(self): # is called by the with statement to enter the runtime context.
        self.start = time.time()
        print ("Getting Started")
        return self

    def __exit__(self, exc_type, exc_value, traceback):# is called when the execution leaves the with code block.
        end = time.time()
        print(f"Elapsed time: {end - self.start}")
        
with Timer():
    print("Timer is on")
    sum(range(1_000_000))       

Getting Started
Timer is on
Elapsed time: 0.022379159927368164


In [55]:
##################################################
#   Custom Class-Based Context Manager                                      
#################################################
class ManagedResource:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"Opening {self.name}")
        return self.name

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Closing {self.name}")
with ManagedResource("database") as db:
    print(f"Working with {db}")

Opening database
Working with database
Closing database


In [3]:
###################################################
#  The same behaviour can be implemented  
# using the contextlib.contextmanager decorator
# Equivalent to  a custom context manager in Python                                   
##################################################
"""
contextlib: Python standard library module for working with context managers.
contextmanager: is a decorator that allows you to write a context manager using a generator function 
instead of defining a full class with __enter__ and __exit__.
"""
from contextlib import contextmanager
@contextmanager
def managed_resource(name):
    print(f"Opening {name}")
    try:
        yield name          # Everything before yield = __enter__
    finally:
        print(f"Closing {name}")  # Everything after yield = __exit__

with managed_resource("database") as db:
    print(f"Working with {db}")

Opening database
Working with database
Closing database


In [5]:
###################################################
#  Context Manager with Exception Handling                                   
##################################################
"""
This context manager is designed to catch and suppress division by zero errors specifically
while allowing all other exceptions to be raised normally.
"""
class SuppressZeroDivision:
    def __enter__(self):
        return self # Returns the context manager instance to be used in the `with` statement.
     """
     - `exc_type`: The exception class
     - `exc_value`: The exception instance
     - `traceback`: The traceback object
    """
    def __exit__(self, exc_type, exc_value, traceback):
        return exc_type is ZeroDivisionError

with SuppressZeroDivision(): # output: ValueError
    raise ValueError("Something went wrong")

print("Continues running")

ValueError: Something went wrong

# 4. Functional Programming
Functional Programming (FP) is a programming paradigm. FP is  a style or way of thinking that models computation as the evaluation of mathematical functions. Rather than issuing a sequence of statements that change the program's state (imperative style), FP describes the answer, not how to compute it step by step.

```python
# --- Imperative (how) ---
temps = [22, 35, 18, 40, 28, 33]

result = []
for t in temps:
    if t > 30:
        result.append(t)

print(result)  # [35, 40, 33]

# --- Functional (what) ---
is_hot = lambda t: t > 30

def keep(predicate, iterable):
    return (x for x in iterable if predicate(x))

result = list(keep(is_hot, temps))

print(result)  # [35, 40, 33]

# --- Pythonic functional (best of both) ---
result = [t for t in temps if t > 30]

print(result)  # [35, 40, 33]
```


### 4.1 Core Principles

The pillars of functional programming are:

| Principle | Meaning |
|---|---|
| **Pure Functions** | Same input → same output, no side effects |
| **Immutability** | Data is never modified in place |
| **First-Class Functions** | Functions are values, passable like any data |
| **Higher-Order Functions** | Functions that take/return other functions |
| **Referential Transparency** | Expressions can be replaced with their result |
| **Declarative Style** | Express *what*, not *how* |


### 4.2 Pure Functions
A **pure function**:
- Returns the same output for the same input, always
- Has **no side effects** (no I/O, no mutation, no global state)

```python
# IMPURE — reads external state, has side effects
tax_rate = 0.2
log = []

def compute_price_impure(price):
    total = price * (1 + tax_rate)  # Reads global variable
    log.append(total)               # Mutates external list
    print(f"Total: {total}")        # I/O side effect
    return total


# PURE — depends only on arguments, has no side effects
def compute_price(price: float, tax_rate: float) -> float:
    return price * (1 + tax_rate)

### 4.3 Immutability

Immutability means **never changing data after creation**. Instead, produce new data with the desired changes.

```python
# Mutable (imperative)
cart = ["apple", "banana"]
cart.append("cherry")      # Mutates cart in place

# Immutable (functional)
cart = ("apple", "banana")
new_cart = cart + ("cherry",)  # Produces a new tuple

# Python's immutable built-ins:
# str, int, float, bool, tuple, frozenset, bytes

In [22]:
# Immutable records with dataclasses
from dataclasses import dataclass, replace

@dataclass(frozen=True)   # frozen=True makes it immutable
class User:
    name: str
    age: int
    email: str

user = User("Alice", 30, "alice@example.com")
# user.age = 31  # Raises FrozenInstanceError!

# To "update", create a new instance
older_user = replace(user, age=31)
print(user)       # User(name='Alice', age=30, ...)
print(older_user) # User(name='Alice', age=31, ...)

User(name='Alice', age=30, email='alice@example.com')
User(name='Alice', age=31, email='alice@example.com')


In [21]:
# Immutable dict operations
original = {"a": 1, "b": 2}
updated = {**original, "c": 3}        # Spread — new dict
updated = original | {"c": 3}         # Python 3.9+ merge operator
print(updated)
filtered = {k: v for k, v in original.items() if k != "a"}  # Filtered new dict
print(filtered)

{'a': 1, 'b': 2, 'c': 3}
{'b': 2}


### 4.4 First-Class

In Python, **functions are first-class objects**  and they can be:
- Assigned to variables
- Stored in data structures
- Passed as arguments
- Returned from other functions

```python
# Functions as values
def add(a, b): return a + b
def multiply(a, b): return a * b

operation = add           # Assign to variable
print(operation(3, 4))   # 7

# Store in data structures
operations = {
    "add": add,
    "mul": multiply,
    "sub": lambda a, b: a - b,
}
print(operations["mul"](3, 4))  # 12

# Functions as arguments — Higher-Order Functions (HOFs)
def apply_twice(func, value):
    return func(func(value))

double = lambda x: x * 2
print(apply_twice(double, 3))   # 12  (3*2=6, 6*2=12)

# Functions as return values
def make_adder(n):
    def adder(x):
        return x + n
    return adder

add5 = make_adder(5)
add10 = make_adder(10)
print(add5(3))   # 8
print(add10(3))  # 13
```

### 4.5 Higher-Order Functions
 `map`, `filter`, `reduce` are the **three fundamental Higher-Order Functions (HOFs)** of functional programming. 
 
 > Please have a look at this [book documentation](https://book.pythontips.com/en/latest/map_filter.html) before checking the starter code.

In [24]:
########################################
# 1. map(func, iterable) — Transform
#######################################

# Apply a function to every element
numbers = [1, 2, 3, 4, 5]

# map returns a lazy iterator
squared = map(lambda x: x**2, numbers)
print(list(squared)) 

[1, 4, 9, 16, 25]


In [40]:
# Multiple iterables
a = [1, 2, 3]
b = [10, 20, 30]
"""
1) This is lamda anonymous function that takes two parameters 
and returns their sum.
2) The `map()` function applies the lambda function to the corresponding elements 
from both lists `a` and `b`. 
3) It processes them in pairs: (1,10), (2,20), (3,30).
"""
sums = list(map(lambda x, y: x + y, a, b))  # The result of `map()` is converted to a list to make it displayable.
print(sums)

[11, 22, 33]


In [42]:
# Real-world: data transformation
records = [
    {"name": "alice", "score": 85},
    {"name": "bob",   "score": 92},
    {"name": "carol", "score": 78},
]
"""
Applies the `title()` method to capitalise the first letter of each word
"""
names = list(map(lambda r: r["name"].title(), records))  # ["Alice", "Bob", "Carol"]
print(names)

['Alice', 'Bob', 'Carol']


In [46]:
############################################
# 2. filter(predicate, iterable) — Select
###########################################
# Keep only elements where predicate returns True
numbers = range(1, 21)
evens = list(filter(lambda x: x % 2 == 0, numbers)) # [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
print(evens)

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


In [47]:
# filter(None, iterable) removes falsy values
mixed = [0, 1, "", "hello", None, [], [1,2], False, True]
truthy = list(filter(None, mixed))  # [1, "hello", [1,2], True]
print(truthy)

[1, 'hello', [1, 2], True]


In [49]:
# Real-world: filter records
employees = [
    {"name": "Alice",   "dept": "Engineering", "salary": 95000},
    {"name": "Bob",     "dept": "Marketing",   "salary": 72000},
    {"name": "Charlie", "dept": "Engineering", "salary": 110000},
    {"name": "Diana",   "dept": "HR",          "salary": 68000},
]

senior_engineers = list(filter(
    lambda e: e["dept"] == "Engineering" and e["salary"] > 90000,
    employees
))
print(senior_engineers) # [{"name": "Alice", ...}, {"name": "Charlie", ...}]

[{'name': 'Alice', 'dept': 'Engineering', 'salary': 95000}, {'name': 'Charlie', 'dept': 'Engineering', 'salary': 110000}]


In [51]:
############################################
# 3. reduce(func, iterable, initial) — Fold/Accumulate
###########################################
from functools import reduce
"""
1. reduce() is part of the functional programming paradigm in Python 
and is particularly useful for operations that need to process all elements in a sequence to produce a single result.
2. reduce() is used to "fold" or "accumulate" a sequence into a single value 
by repeatedly applying a binary function.
"""
numbers = [1, 2, 3, 4, 5]
"""
- It takes a binary function (a function that takes two arguments)
   - It applies this function to the first two items of the iterable
   - Then it applies the function to the result and the next item
   - This process continues until all items are processed
Example:
- Step 1: Apply the function to 1 and 2, the result is 3
   - Step 2: Apply the function to 3 (previous result) and 3, the result is 6
   - Step 3: Apply the function to 6 and 4, the result is 10
   - Step 4: Apply the function to 10 and 5, the result is 15
   - Final result: 15
"""
# Sum 
total = reduce(lambda acc, x: acc + x, numbers, 0)  # 15
print(total)

15


In [52]:
# Product
product = reduce(lambda acc, x: acc * x, numbers, 1)  # 120
print(product)

# Maximum
maximum = reduce(lambda acc, x: acc if acc > x else x, numbers)  # 5
print(maximum)

120
5


In [54]:
############################################
# 4. Chaining map → filter → reduce
###########################################
"""
We often want to perform multiple steps in sequence.

Example problem:

Compute the sum of the squares of even numbers from a list.

Steps:

1. Filter even numbers

2. Square each number

3. Sum the result
"""
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6]

evens = filter(lambda x: x % 2 == 0, numbers)

squares = map(lambda x: x**2, evens)

result = reduce(lambda a, b: a + b, squares)

print(result)

56


In [55]:
############################################
# Equivalent to chaining above in One Line
###########################################
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6]

result = reduce(
    lambda a, b: a + b,
    map(
        lambda x: x**2,
        filter(lambda x: x % 2 == 0, numbers)
    )
)
print(result)

56


### Now we are done with week 1's lab content 🎉🎉🎉 ##
![Week 1](gen.gif)
### Let's navigate to Exercise's notebook ⬇️⬇️⬇️
[Exercise Notebook](Week1_Exercise.ipynb)